# Domain adaptation in classical machine learning

## Research question and hypotheses

For Logistic Regression, linear SVM and Random Forest, when does importance weighting or correlation alignment (CORAL) improve or worsen held-out target macro F1 relative to source-only training under changes in gas sensor collection batch, smartphone participants and product review category, and what accuracy and computation-time trade-offs accompany these changes?

For each transfer pair and classifier, define the paired effect as `delta = macro_F1(adapted) - macro_F1(source_only)` on identical target test IDs. **H1:** importance weighting yields a positive delta. **H2:** CORAL yields a positive delta. Each hypothesis is assessed separately for every pair and classifier; zero or negative effects are valid outcomes. Weighting is expected to depend on source–target overlap, and CORAL on useful mean/covariance alignment; these are diagnostic interpretations, not guaranteed improvements or additional selection criteria.

## Domains and prediction tasks

| Shift / dataset | Labelled source | Unlabelled target adaptation | Held-out target evaluation | Transfer pairs |
|---|---|---|---|---|
| Time / UCI Gas Sensor Array Drift | All batch 1 rows; predict one of six gases | For each batch 2–10 separately, shuffle sorted IDs with the run seed and take the first floor(N/2) rows | Remaining rows of that same later batch | 9: batch 1 → 2, 3, 4, 5, 6, 7, 8, 9, 10 |
| People / UCI HAR Using Smartphones | All 21 official training participants; predict one of six activities | Shuffle the nine sorted official test participant IDs with the seed; take four whole participants | Other five whole official test participants; keep each person's windows together | 1 participant-transfer design |
| Category / Amazon Multi-Domain Sentiment | Books positive and negative reviews from `processed_acl.tar.gz`; predict sentiment | Electronics unlabelled reviews | Electronics positive and negative reviews | 1: Books → Electronics |

There are **11 transfer pairs**. Restart from Gas batch 1 for every later batch. Roles are disjoint; assign target roles without task labels. Amazon release roles stay fixed across seeds, subject to a feature-only duplicate audit: source takes priority over target test, which takes priority over adaptation. Strip `#label#` from every Amazon input dictionary and exclude domain, path, polarity filename and rating from predictors. Batch and participant identifiers are grouping metadata, not predictors.

## Registered comparison matrix

Apply every row below to every transfer pair and seed. All rows predict the same held-out IDs within that pair and seed.

| Configuration | Classifier | Training condition |
|---|---|---|
| LR-SO | Logistic Regression | Source only |
| LR-IW | Logistic Regression | Importance weighting |
| LR-CORAL | Logistic Regression | CORAL |
| SVM-SO | Linear SVM (`LinearSVC`) | Source only |
| SVM-IW | Linear SVM (`LinearSVC`) | Importance weighting |
| SVM-CORAL | Linear SVM (`LinearSVC`) | CORAL |
| RF-SO | Random Forest | Source only |
| RF-IW | Random Forest | Importance weighting |
| RF-CORAL | Random Forest | CORAL |

The majority-class control is separate from these nine main configurations.

| Control | Fit rule | Rows per transfer pair and seed |
|---|---|---|
| Source majority | Predict the most frequent source task label; break ties by sorted class ID | 1, independently of classifier |

| Condition | Permitted information for fitting |
|---|---|
| Source only | Source features and source task labels |
| Importance weighting | Source features/task labels plus unlabelled adaptation features; source/adaptation domain membership trains the domain classifier |
| CORAL | Source features/task labels plus unlabelled adaptation features for mean/covariance alignment |
| Majority control | Source task-label counts only |

No method uses adaptation task labels. Held-out target features are used only for frozen transformation and prediction; held-out task labels are read only by the final evaluator after predictions are frozen. Target scores never select settings.

## Shared representation, settings and ablation

Use seeds **11, 23, 37, 53, 71**. Reuse each split manifest and ordered test-ID list across methods. Seeds control eligible splits, source folds, randomized reduction and estimators; Amazon's released task roles do not change.

Fit ordinary preprocessing on source training rows only, inside each source-validation fold: imputation when needed and standardization for Gas/HAR; source dictionary, TF-IDF, truncated SVD and standardization for Amazon. All branches, including Random Forest, share the same starting representation. For Amazon use `k = min(100, n_source_rows - 1, n_source_features - 1)` within each fit.

Tune each classifier by source-only three-fold macro-F1 validation, with participant-disjoint stratified group folds for HAR and shuffled stratified folds otherwise. Reuse the selected settings in all three branches. Logistic Regression and LinearSVC use `C = [0.1, 1, 10]`; Random Forest uses 200 trees, square-root feature sampling and `max_depth = [12, None]`. Resolve exact tuning ties by the first candidate. Use `class_weight=None` and one CPU thread per estimator. Source-only tuning is a limitation of this comparison.

For importance weighting, use an unweighted logistic domain classifier (`C=1`) and seeded three-fold out-of-fold source probabilities. Clamp target probability `p` to `[1e-6, 1-1e-6]`; calculate `(p / (1-p)) * (n_source_fit / n_adapt_fit)` using domain-training counts for that fold. Cap raw weights at 10, then normalize to mean one. Record overlap/weight diagnostics and effective sample size. For CORAL, align source means and covariances to adaptation data in float64, adding `1e-3 I` to each covariance; leave target features in their original shared coordinates.

**Registered ablation:** remove only the upper weight cap, retaining probability safeguards, mean normalization, source settings and matched splits. Compare uncapped minus capped importance weighting separately for all three classifiers. Freeze main and ablation predictions before opening target test labels.

**Planned inventory:** `11 × 5 × 3 × 3 = 495` main rows; `11 × 5 = 55` majority-control rows; `11 × 5 × 3 = 165` additional uncapped-weight rows. These are planned counts, not observed results. Exclude pilots, tuning and retries from final totals; check configuration identities as well as counts.

## Outcomes and interpretation

**Primary:** target macro F1 over the sorted classes with positive true support in the original test split, using zero for an undefined class F1. Use the same label set for all methods and bootstrap draws within that split. Report per-class support and full-vocabulary confusion matrices.

**Secondary:** target accuracy and wall-clock seconds on the same hosted Colab CPU. Total compute time is shared preprocessing fit/transform + adaptation fit/transform + classifier fit + prediction. Charge measured shared stages to each branch that reuses them. Exclude downloading, file I/O, tuning, scoring and bootstrapping; report tuning time separately. Positive accuracy differences indicate improvement; positive time differences indicate added cost.

Compare each adapted method minus its matching source-only classifier, by transfer pair and seed. Report per-seed effects and five-seed mean ± sample SD for scores and paired effects. Keep transfer pairs visible so nine Gas transfers do not dominate the other datasets.

For **seed 11 only**, use a paired 95% percentile bootstrap interval for macro-F1 delta: 2,000 draws with resampling seed 2026 and identical resampled units for both methods. Resample whole participants for HAR, unique retained reviews for Amazon and measurements within each Gas target batch. HAR has only five evaluation clusters; Gas intervals are conditional on batch and assume independent measurements, so temporal dependence may make them too narrow. Repeated splits and shared source data are dependent.

Predeclare this descriptive rule: an interval entirely above zero supports a **gain**; entirely below zero supports a **loss / negative transfer**; an interval containing or touching zero is **inconclusive**, not evidence of equivalence. No practical-effect threshold or multiplicity-adjusted claim is prespecified. The seed-11 interval does not describe uncertainty in the five-seed mean; report their signs separately and disclose disagreement. Apply the same interval rule to uncapped-minus-capped ablation effects. No experimental result is asserted here.